# Lab 14 — Flow Orchestration / CrewAI Flows

Use case: support extraction stateful flow for Ukrainian admin-service messages.

Domain: **Дія**, **єВідновлення**, **ЦНАП**, **Паспортний сервіс**, **Нотаріус**.

The flow implements five auditable stages: **ingest → route → execute → validate → (fallback) → export**.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    import subprocess, pathlib
    if not pathlib.Path('/content/nlp').exists():
        subprocess.check_call(['git', 'clone', 'https://github.com/velotsuraptor/nlp_labs.git', '/content/nlp'])
'repo ready'

## 1. Install Dependencies

In [ ]:
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'jsonschema>=4.0.0', 'pandas>=1.3.0'])
else:
    try:
        import jsonschema
        import pandas
        print(f'jsonschema {jsonschema.__version__}, pandas {pandas.__version__} — ready')
    except ImportError as e:
        print(f'Missing dependency: {e}. Run: pip install -r requirements.txt')

## 2. Test Cases

In [ ]:
import json
import pathlib
import sys

def find_project_root(marker: str = 'project_lab14') -> pathlib.Path:
    """Walk up from this notebook's location (or CWD) to find the project root."""
    candidates = [
        pathlib.Path('/content/nlp'),          # Colab
        pathlib.Path(__file__).resolve().parent if '__file__' in dir() else pathlib.Path.cwd(),
        pathlib.Path.cwd(),
    ]
    for base in candidates:
        # Walk up from base
        for p in [base, *base.parents]:
            if (p / marker).exists():
                return p
    # Fallback: CWD
    return pathlib.Path.cwd()

REPO_ROOT = find_project_root()
LAB_ROOT = REPO_ROOT / 'project_lab14'
SRC_DIR = LAB_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

print(f'LAB_ROOT = {LAB_ROOT}')
print(f'SRC_DIR  = {SRC_DIR}')

tc_path = LAB_ROOT / 'data' / 'sample' / 'test_cases_lab14.jsonl'
test_cases = [json.loads(line) for line in tc_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print(f'\nLoaded {len(test_cases)} test cases')
print('First case:')
print(json.dumps(test_cases[0], ensure_ascii=False, indent=2))

## 3. Flow State Definition

In [ ]:
import dataclasses

try:
    from flow_state import FlowState
except ImportError:
    from src.flow_state import FlowState

print('FlowState fields:')
for f in dataclasses.fields(FlowState):
    default = f.default if f.default is not dataclasses.MISSING else (
        f'factory: {f.default_factory.__name__}' if f.default_factory is not dataclasses.MISSING else 'required'
    )
    print(f'  {f.name:25s}  {f.type}  (default: {default})')

## 4. Memory / Knowledge Policy

**What is stored in FlowState:**
Each `FlowState` instance holds only the data for a single case during its pipeline run. It is created fresh for each `run()` call and discarded after export. Nothing persists between cases.

**What is NOT stored:**
- No prior conversation turns, user session data, or message history.
- No cross-case context: case N has no access to results of case N-1.
- No user identity or credentials.

**Knowledge resources used:**
- Ingest: normalization table (hardcoded replacements).
- Route: `SERVICE_KEYWORDS`, `MONEY_RE`, document/issue keyword lists.
- Execute: `_SERVICE_PATTERNS`, `_ISSUE_PATTERNS`, amount/date/location regexes.
- Validate: `SERVICE_ENUM`, `ISSUE_ENUM`, required fields from `ROUTES`.
- Fallback: `_SERVICE_REPAIR_PATTERNS`.

All knowledge resources are **static module-level constants** — no external KB, vector store, or LLM call.

**Logger persistence:**
`FlowLogger.save_jsonl(path)` writes all in-memory log records to a JSONL file — the only persistent artifact produced by the flow.

## 5. Knowledge Resources / Schemas

In [ ]:
try:
    from executor import SERVICE_ENUM, ISSUE_ENUM
    from router import ROUTES
except ImportError:
    from src.executor import SERVICE_ENUM, ISSUE_ENUM
    from src.router import ROUTES

print('SERVICE_ENUM:', SERVICE_ENUM)
print()
print('ISSUE_ENUM:', ISSUE_ENUM)
print()
print('ROUTES:')
for route_name, cfg in ROUTES.items():
    print(f'  {route_name}: schema={cfg["schema"]}, required={cfg["required_fields"]}')

## 6. Ingest Step

In [ ]:
try:
    from executor import normalize
except ImportError:
    from src.executor import normalize

# Demo: ingest on case_001
demo_case = test_cases[0]
raw_text = demo_case['input']
clean_text = normalize(raw_text)

state_demo = FlowState(case_id=demo_case['case_id'])
state_demo.raw_text = raw_text
state_demo.clean_text = clean_text
state_demo.status = 'ingested'
state_demo.steps.append({'step': 'ingest', 'status': 'ok', 'clean_text_length': len(clean_text)})

print('--- Ingest step demo ---')
print(f'case_id    : {state_demo.case_id}')
print(f'raw_text   : {state_demo.raw_text}')
print(f'clean_text : {state_demo.clean_text}')
print(f'status     : {state_demo.status}')
print(f'steps      : {state_demo.steps}')

## 7. Route Step

In [ ]:
try:
    from router import route as route_fn
except ImportError:
    from src.router import route as route_fn

routing = route_fn(state_demo.clean_text)
print('Route output:')
print(json.dumps(routing, ensure_ascii=False, indent=2))

state_demo.route = routing['route']
state_demo.schema_name = routing['schema_name']
state_demo.routing_reason = routing['routing_reason']
state_demo.steps.append({'step': 'route', 'status': 'ok', **routing})
print(f'\nstate.route = {state_demo.route}')

## 8. Execute Step

In [ ]:
try:
    from executor import execute
except ImportError:
    from src.executor import execute

exec_output = execute(state_demo.route, state_demo.clean_text)
state_demo.execute_output = exec_output
state_demo.steps.append({
    'step': 'execute', 'status': 'ok',
    'primary_service': exec_output.get('primary_service'),
    'issue_type': exec_output.get('issue_type'),
    'confidence': exec_output.get('confidence'),
})

print('Execute output:')
print(json.dumps(exec_output, ensure_ascii=False, indent=2))

## 9. Validate Step

In [ ]:
try:
    from validator import validate
except ImportError:
    from src.validator import validate

required_fields = ROUTES.get(state_demo.route, {}).get('required_fields', [])
vr = validate(state_demo.route, state_demo.execute_output, required_fields)
state_demo.validation_result = vr

print('Validation result:')
print(json.dumps(vr, ensure_ascii=False, indent=2))

## 10. Fallback Logic

In [ ]:
try:
    from fallback import fallback as fallback_fn
except ImportError:
    from src.fallback import fallback as fallback_fn

# Use case_009_ambiguous — ambiguous service → validation warning → triggers export_with_warning
# Show fallback on case_006 which actually gets repaired
case_006 = next(c for c in test_cases if c['case_id'] == 'case_006_fallback_helps')
clean_006 = normalize(case_006['input'])
exec_006 = execute('support_extraction', clean_006)

# Simulate a validation failure for demo (force primary_service=unknown)
exec_006_fail = dict(exec_006, primary_service='unknown')
vr_006 = validate('support_extraction', exec_006_fail, ['primary_service', 'issue_type'])

fb_result = fallback_fn('support_extraction', clean_006, vr_006)

print(f'Input text : {case_006["input"]}')
print(f'Clean text : {clean_006}')
print()
print('Validation result (simulated failure):')
print(json.dumps(vr_006, ensure_ascii=False, indent=2))
print()
print('Fallback result:')
print(json.dumps(fb_result, ensure_ascii=False, indent=2))

## 11. Export Step

In [ ]:
try:
    from exporter import export_result
except ImportError:
    from src.exporter import export_result

# Finalize state_demo status
state_demo.status = 'exported' if not vr.get('warnings') else 'exported_with_warning'
state_demo.warnings = vr.get('warnings', [])
state_demo.steps.append({'step': 'export', 'status': 'ok', 'final_status': state_demo.status})

export = export_result(state_demo.to_dict())
print('Export result:')
print(json.dumps(export, ensure_ascii=False, indent=2))

## 12. Ad-hoc Baseline (Without Flow)

In [ ]:
# Ad-hoc baseline: calls execute() directly, no stage tracking, no fallback, no structured export on failure

def adhoc_extract(text: str) -> dict:
    """Simple ad-hoc extraction without flow orchestration."""
    try:
        clean = normalize(text)
        result = execute('support_extraction', clean)
        return {'status': 'ok', 'output': result}
    except Exception as e:
        return {'status': 'error', 'error': str(e), 'output': None}

adhoc_cases = [
    test_cases[0],   # case_001 — clean input
    test_cases[6],   # case_007 — no service, fallback would fail
    test_cases[9],   # case_010 — empty input
]

print('Ad-hoc baseline results (no flow):')
for c in adhoc_cases:
    r = adhoc_extract(c['input'])
    print(f'\n[{c["case_id"]}]')
    print(f'  status : {r["status"]}')
    if r['output']:
        print(f'  primary_service: {r["output"].get("primary_service")}')
        print(f'  issue_type     : {r["output"].get("issue_type")}')
        print(f'  confidence     : {r["output"].get("confidence")}')
    print(f'  steps_tracked  : No')
    print(f'  fallback       : No')
    print(f'  structured_fail: No')

## 13. Run All 12 Test Cases Through SupportExtractionFlow

In [ ]:
import pandas as pd

try:
    from flow import SupportExtractionFlow
except ImportError:
    from src.flow import SupportExtractionFlow

flow = SupportExtractionFlow()

results = []
for tc in test_cases:
    export = flow.run(tc['case_id'], tc['input'])
    results.append({
        'case_id': tc['case_id'],
        'route': export.get('route', ''),
        'final_status': export.get('status', ''),
        'fallback_triggered': export.get('fallback_triggered', False),
        'warnings_count': len(export.get('warnings', [])),
        'needs_manual_review': export.get('needs_manual_review', False),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 14. Flow Logs

In [ ]:
logs_path = LAB_ROOT / 'docs' / 'flow_logs_lab14.jsonl'
flow.logger.save_jsonl(str(logs_path))
print(f'Saved {len(flow.logger.records)} log records to {logs_path}')

# Load and display
log_records = [json.loads(line) for line in logs_path.read_text(encoding='utf-8').splitlines() if line.strip()]

logs_df = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'route': r.get('route', ''),
        'final_status': r.get('final_status', ''),
        'fallback_triggered': r.get('fallback_triggered', False),
        'n_steps': len(r.get('steps', [])),
        'n_warnings': len(r.get('warnings', [])),
        'n_errors': len(r.get('errors', [])),
    }
    for r in log_records
])

print('\nFirst 5 log records:')
print(logs_df.head(5).to_string(index=False))

## 15. Metrics

In [ ]:
n = len(results)
n_completed = sum(1 for r in results if r['final_status'] not in ('', None))
n_validation_pass = sum(1 for r in results if r['final_status'] in ('exported', 'exported_with_warning'))
n_fallback = sum(1 for r in results if r['fallback_triggered'])
n_fallback_success = sum(1 for r in results if r['final_status'] in ('fallback_repaired',))
n_safe_failure = sum(1 for r in results if r['final_status'] == 'safe_failure')
n_export_valid = sum(1 for r in results if r['final_status'] not in ('', None))
avg_steps = logs_df['n_steps'].mean() if not logs_df.empty else 0
avg_warnings = results_df['warnings_count'].mean()

metrics = {
    'total_cases': n,
    'flow_completion_rate': round(n_completed / n, 3),
    'validation_pass_rate': round(n_validation_pass / n, 3),
    'fallback_activation_rate': round(n_fallback / n, 3),
    'fallback_success_rate': round(n_fallback_success / max(n_fallback, 1), 3),
    'safe_failure_rate': round(n_safe_failure / n, 3),
    'export_valid_rate': round(n_export_valid / n, 3),
    'avg_steps_per_case': round(avg_steps, 2),
    'avg_warnings_per_case': round(avg_warnings, 2),
}

print('Flow metrics:')
for k, v in metrics.items():
    print(f'  {k:35s}: {v}')

## 16. Error Analysis

In [ ]:
error_path = LAB_ROOT / 'docs' / 'error_cases_lab14.json'
error_cases = json.loads(error_path.read_text(encoding='utf-8'))

error_df = pd.DataFrame([
    {
        'case_id': e['case_id'],
        'error_category': e['error_category'],
        'final_status': e['final_status'],
        'possible_fix': e['possible_fix'][:60] + '...' if len(e['possible_fix']) > 60 else e['possible_fix'],
    }
    for e in error_cases
])

print('Error analysis:')
print(error_df.to_string(index=False))

## 17. Compare With Ad-hoc Pipeline

In [ ]:
compare_ids = ['case_001_simple_service', 'case_007_fallback_fails', 'case_010_manual_review']
compare_cases = [tc for tc in test_cases if tc['case_id'] in compare_ids]

# Re-run flow for comparison cases on a fresh flow instance
flow2 = SupportExtractionFlow()
comparison = []
for tc in compare_cases:
    adhoc = adhoc_extract(tc['input'])
    flow_export = flow2.run(tc['case_id'], tc['input'])

    adhoc_status = 'ok' if adhoc['status'] == 'ok' and adhoc.get('output') else 'silent_failure'
    flow_status = flow_export.get('status', 'unknown')

    comparison.append({
        'case_id': tc['case_id'],
        'adhoc_status': adhoc_status,
        'flow_status': flow_status,
        'adhoc_has_fallback': False,
        'flow_has_fallback': flow_export.get('fallback_triggered', False),
        'adhoc_structured_failure': False,
        'flow_structured_failure': flow_status == 'safe_failure',
    })

cmp_df = pd.DataFrame(comparison)
print('Ad-hoc vs Flow comparison:')
print(cmp_df.to_string(index=False))

## 18. Generate Docs

In [ ]:
docs_dir = LAB_ROOT / 'docs'
docs_dir.mkdir(parents=True, exist_ok=True)

# --- Audit summary ---
audit_text = f"""# Lab 14 — Audit Summary (Generated)

## Run Metrics

| Metric | Value |
|--------|-------|
| Total cases | {metrics['total_cases']} |
| Flow completion rate | {metrics['flow_completion_rate']} |
| Validation pass rate | {metrics['validation_pass_rate']} |
| Fallback activation rate | {metrics['fallback_activation_rate']} |
| Fallback success rate | {metrics['fallback_success_rate']} |
| Safe failure rate | {metrics['safe_failure_rate']} |
| Export valid rate | {metrics['export_valid_rate']} |
| Avg steps per case | {metrics['avg_steps_per_case']} |
| Avg warnings per case | {metrics['avg_warnings_per_case']} |

## Status Distribution
"""
status_counts = results_df['final_status'].value_counts()
for status, count in status_counts.items():
    audit_text += f'- {status}: {count}\n'

(docs_dir / 'audit_summary_lab14.md').write_text(audit_text, encoding='utf-8')
print(f'Wrote: {docs_dir / "audit_summary_lab14.md"}')

# --- Flow notes (pointer to static file) ---
flow_notes_src = docs_dir / 'flow_notes_lab14.md'
if flow_notes_src.exists():
    print(f'Flow notes already exist: {flow_notes_src}')

# --- Memory policy (pointer to static file) ---
mem_policy_src = docs_dir / 'memory_policy_lab14.md'
if mem_policy_src.exists():
    print(f'Memory policy already exists: {mem_policy_src}')

# --- Error cases JSON (re-save from loaded data) ---
error_out = docs_dir / 'error_cases_lab14.json'
error_out.write_text(json.dumps(error_cases, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Wrote: {error_out}')

# --- Flow logs JSONL ---
print(f'Flow logs: {logs_path}')

print('\nAll docs files:')
for f in sorted(docs_dir.iterdir()):
    print(f'  {f}')